In [ ]:
!pip install -q pyspark pyarrow pandas plotly kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.9 MB/s eta 0:00:00


In [ ]:
import os

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import HTML, display
from google.colab import drive
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [ ]:
drive.mount(
    "/content/drive",
    force_remount=True
)

Mounted at /content/drive


In [ ]:
spark = (
    SparkSession.builder
    .appName("TechChallenge_Dashboard")
    .getOrCreate()
)

BASE_PATH = (
    "/content/drive/MyDrive/"
    "TechChallenge_Fase2"
)

GOLD_PATH = f"{BASE_PATH}/data/gold"
OUTPUT_PATH = f"{BASE_PATH}/outputs"
DASHBOARD_PATH = f"{OUTPUT_PATH}/dashboard"

os.makedirs(
    DASHBOARD_PATH,
    exist_ok=True
)



In [ ]:
def carregar_gold(nome_tabela):
    caminho = f"{GOLD_PATH}/{nome_tabela}"

    if not os.path.exists(caminho):
        raise FileNotFoundError(
            f"Data Mart não encontrado: {caminho}"
        )

    return spark.read.parquet(caminho)

In [ ]:
gold_municipio = carregar_gold(
    "desempenho_municipio"
)

gold_uf = carregar_gold(
    "desempenho_uf"
)

gold_meta_municipio = carregar_gold(
    "meta_municipio"
)

gold_meta_uf = carregar_gold(
    "meta_uf"
)

gold_meta_brasil = carregar_gold(
    "meta_brasil"
)

gold_perfil_alunos = carregar_gold(
    "perfil_alunos"
)

gold_kpis = carregar_gold(
    "kpis"
)



In [ ]:
data_marts = {
    "desempenho_municipio": gold_municipio,
    "desempenho_uf": gold_uf,
    "meta_municipio": gold_meta_municipio,
    "meta_uf": gold_meta_uf,
    "meta_brasil": gold_meta_brasil,
    "perfil_alunos": gold_perfil_alunos,
    "kpis": gold_kpis
}

for nome, dataframe in data_marts.items():
    print(
        f"{nome}: "
        f"{dataframe.count():,} registros e "
        f"{len(dataframe.columns)} colunas"
    )

desempenho_municipio: 23,995 registros e 24 colunas
desempenho_uf: 145 registros e 24 colunas
meta_municipio: 10,704 registros e 21 colunas
meta_uf: 81 registros e 23 colunas
meta_brasil: 3 registros e 22 colunas
perfil_alunos: 79,273 registros e 24 colunas
kpis: 30 registros e 7 colunas


In [ ]:
ANO_REFERENCIA = (
    gold_municipio
    .select(
        F.max("ano").alias("ano")
    )
    .first()["ano"]
)

print(
    f"Ano de referência do dashboard: "
    f"{ANO_REFERENCIA}"
)

Ano de referência do dashboard: 2024


In [ ]:
kpis_pd = (
    gold_kpis
    .filter(
        F.col("ano_referencia")
        == ANO_REFERENCIA
    )
    .orderBy(
        "grupo",
        "indicador"
    )
    .toPandas()
)

kpis_pd

,grupo,indicador,valor_numerico,valor_texto,unidade,data_processamento,ano_referencia
0,Brasil,Cumprimento da meta nacional,98.83,None,percentual,2026-07-14 05:52:25.536397,2024
1,Brasil,Meta nacional de 2024,59.90,None,percentual,2026-07-14 05:52:25.536397,2024
2,Brasil,Taxa nacional de alfabetização,59.20,None,percentual,2026-07-14 05:52:25.536397,2024
3,Cobertura,Municípios presentes na base de alunos,5519.00,None,municípios,2026-07-14 05:52:25.536397,2024
4,Cobertura,Total de UFs distintas,25.00,None,UFs,2026-07-14 05:52:25.536397,2024
5,Cobertura,Total de escolas representadas,42497.00,None,escolas,2026-07-14 05:52:25.536397,2024
6,Cobertura,Total de municípios distintos,5516.00,None,municípios,2026-07-14 05:52:25.536397,2024
7,Desempenho UF,Média da taxa de alfabetização,57.36,None,percentual,2026-07-14 05:52:25.536397,2024
8,Desempenho UF,Média de Português,746.29,None,pontos,2026-07-14 05:52:25.536397,2024
9,Desempenho UF,Média do índice geral,65.99,None,índice,2026-07-14 05:52:25.536397,2024


In [ ]:
def obter_kpi(
    grupo,
    indicador,
    valor_padrao=None
):
    filtro = kpis_pd.loc[
        (
            kpis_pd["grupo"] == grupo
        )
        & (
            kpis_pd["indicador"] == indicador
        )
    ]

    if filtro.empty:
        return valor_padrao

    valor = filtro.iloc[0]["valor_numerico"]

    if pd.isna(valor):
        return valor_padrao

    return float(valor)

In [ ]:
total_municipios = obter_kpi(
    "Cobertura",
    "Total de municípios distintos",
    0
)

total_ufs = obter_kpi(
    "Cobertura",
    "Total de UFs distintas",
    0
)

total_escolas = obter_kpi(
    "Cobertura",
    "Total de escolas representadas",
    0
)

media_alfabetizacao_municipal = obter_kpi(
    "Desempenho municipal",
    "Média da taxa de alfabetização",
    0
)

media_portugues = obter_kpi(
    "Desempenho municipal",
    "Média de Português",
    0
)

percentual_excelente = obter_kpi(
    "Desempenho municipal",
    "Registros com desempenho excelente",
    0
)

total_alunos_representados = obter_kpi(
    "Perfil de alunos",
    "Total de registros de alunos representados",
    0
)

taxa_presenca = obter_kpi(
    "Perfil de alunos",
    "Taxa ponderada de presença",
    0
)

taxa_alfabetizacao_alunos = obter_kpi(
    "Perfil de alunos",
    "Taxa ponderada de alfabetização",
    0
)

proficiencia_media = obter_kpi(
    "Perfil de alunos",
    "Proficiência média ponderada",
    0
)

percentual_acima_corte = obter_kpi(
    "Perfil de alunos",
    "Grupos acima do ponto de corte de 743",
    0
)

In [ ]:
def formatar_numero(
    valor,
    casas=2
):
    if valor is None:
        return "N/D"

    return (
        f"{valor:,.{casas}f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

In [ ]:
cards_gerais = f"""
<h2>Visão Geral — {ANO_REFERENCIA}</h2>

<div style="
    display:grid;
    grid-template-columns:repeat(3, 1fr);
    gap:16px;
    margin:20px 0;
">
    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Municípios</h4>
        <h2>{formatar_numero(total_municipios, 0)}</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>UFs</h4>
        <h2>{formatar_numero(total_ufs, 0)}</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Escolas representadas</h4>
        <h2>{formatar_numero(total_escolas, 0)}</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Média de alfabetização</h4>
        <h2>{formatar_numero(media_alfabetizacao_municipal)}%</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Média de Português</h4>
        <h2>{formatar_numero(media_portugues)}</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Desempenho excelente</h4>
        <h2>{formatar_numero(percentual_excelente)}%</h2>
    </div>
</div>
"""

display(
    HTML(cards_gerais)
)

In [ ]:
cards_alunos = f"""
<h2>Perfil Agregado dos Alunos — {ANO_REFERENCIA}</h2>

<div style="
    display:grid;
    grid-template-columns:repeat(3, 1fr);
    gap:16px;
    margin:20px 0;
">
    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Registros representados</h4>
        <h2>{formatar_numero(total_alunos_representados, 0)}</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Taxa de presença</h4>
        <h2>{formatar_numero(taxa_presenca)}%</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Taxa de alfabetização</h4>
        <h2>{formatar_numero(taxa_alfabetizacao_alunos)}%</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Proficiência média</h4>
        <h2>{formatar_numero(proficiencia_media)}</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Grupos acima de 743</h4>
        <h2>{formatar_numero(percentual_acima_corte)}%</h2>
    </div>

    <div style="padding:20px;border:1px solid #ddd;border-radius:12px;text-align:center;">
        <h4>Ponto de corte</h4>
        <h2>743</h2>
    </div>
</div>
"""

display(
    HTML(cards_alunos)
)

In [ ]:
distribuicao_municipal_pd = (
    gold_municipio
    .filter(
        F.col("ano") == ANO_REFERENCIA
    )
    .groupBy(
        "nivel_desempenho"
    )
    .agg(
        F.count("*").alias("quantidade")
    )
    .orderBy(
        F.desc("quantidade")
    )
    .toPandas()
)

In [ ]:
fig_distribuicao_municipal = px.pie(
    distribuicao_municipal_pd,
    names="nivel_desempenho",
    values="quantidade",
    hole=0.45,
    title=(
        "Distribuição do desempenho municipal "
        f"— {ANO_REFERENCIA}"
    )
)

fig_distribuicao_municipal.update_traces(
    textinfo="percent+label"
)

fig_distribuicao_municipal.show()

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




In [ ]:
uf_total = (
    gold_uf
    .filter(
        (F.col("ano") == ANO_REFERENCIA)
        & (F.col("categoria_rede") == "Total")
    )
)

quantidade_uf_total = (
    uf_total
    .select("sigla_uf")
    .distinct()
    .count()
)

if quantidade_uf_total >= 20:
    uf_ranking = uf_total
    descricao_rede_uf = "Total"
else:
    janela_melhor_uf = (
        F.row_number().over(
            __import__(
                "pyspark.sql.window",
                fromlist=["Window"]
            ).Window
            .partitionBy("sigla_uf")
            .orderBy(
                F.col("indice_geral").desc_nulls_last()
            )
        )
    )

    uf_ranking = (
        gold_uf
        .filter(
            F.col("ano") == ANO_REFERENCIA
        )
        .withColumn(
            "ordem_uf",
            janela_melhor_uf
        )
        .filter(
            F.col("ordem_uf") == 1
        )
    )

    descricao_rede_uf = "melhor registro por UF"

In [ ]:
from pyspark.sql.window import Window

janela_melhor_uf = (
    Window
    .partitionBy("sigla_uf")
    .orderBy(
        F.col("indice_geral").desc_nulls_last()
    )
)

if quantidade_uf_total < 20:
    uf_ranking = (
        gold_uf
        .filter(
            F.col("ano") == ANO_REFERENCIA
        )
        .withColumn(
            "ordem_uf",
            F.row_number().over(
                janela_melhor_uf
            )
        )
        .filter(
            F.col("ordem_uf") == 1
        )
    )

In [ ]:
ranking_uf_pd = (
    uf_ranking
    .select(
        "sigla_uf",
        "categoria_rede",
        "indice_geral",
        "taxa_alfabetizacao",
        "media_portugues"
    )
    .orderBy(
        F.desc("indice_geral")
    )
    .limit(10)
    .toPandas()
)

fig_ranking_uf = px.bar(
    ranking_uf_pd.sort_values(
        "indice_geral",
        ascending=True
    ),
    x="indice_geral",
    y="sigla_uf",
    orientation="h",
    text="indice_geral",
    hover_data={
        "categoria_rede": True,
        "taxa_alfabetizacao": ":.2f",
        "media_portugues": ":.2f"
    },
    title=(
        "Top 10 UFs por índice geral "
        f"— {ANO_REFERENCIA}"
    )
)

fig_ranking_uf.update_traces(
    textposition="outside"
)

fig_ranking_uf.show()

In [ ]:
comparacao_uf_pd = (
    gold_meta_uf
    .filter(
        F.col("ano") == ANO_REFERENCIA
    )
    .select(
        "sigla_uf",
        "taxa_alfabetizacao",
        "meta_alfabetizacao_2024",
        "diferenca_meta_2024",
        "status_meta_2024"
    )
    .orderBy(
        F.desc("taxa_alfabetizacao")
    )
    .toPandas()
)

comparacao_long = comparacao_uf_pd.melt(
    id_vars=[
        "sigla_uf",
        "status_meta_2024"
    ],
    value_vars=[
        "taxa_alfabetizacao",
        "meta_alfabetizacao_2024"
    ],
    var_name="indicador",
    value_name="percentual"
)

comparacao_long["indicador"] = (
    comparacao_long["indicador"]
    .replace({
        "taxa_alfabetizacao": "Taxa observada",
        "meta_alfabetizacao_2024": "Meta de 2024"
    })
)

In [ ]:
fig_comparacao_uf = px.bar(
    comparacao_long,
    x="sigla_uf",
    y="percentual",
    color="indicador",
    barmode="group",
    title=(
        "Taxa de alfabetização versus meta "
        f"por UF — {ANO_REFERENCIA}"
    )
)

fig_comparacao_uf.update_layout(
    xaxis_title="UF",
    yaxis_title="Percentual"
)

fig_comparacao_uf.show()

In [ ]:
evolucao_alunos = (
    gold_perfil_alunos
    .groupBy("ano")
    .agg(
        F.sum(
            "total_registros"
        ).alias(
            "total_registros"
        ),

        F.sum(
            "total_presentes"
        ).alias(
            "total_presentes"
        ),

        F.sum(
            "total_alfabetizados"
        ).alias(
            "total_alfabetizados"
        ),

        F.sum(
            "total_cadernos_preenchidos"
        ).alias(
            "total_cadernos_preenchidos"
        ),

        F.sum(
            F.col("proficiencia_media")
            * F.col("total_registros")
        ).alias(
            "soma_proficiencia_ponderada"
        )
    )

    .withColumn(
        "taxa_presenca",
        F.round(
            (
                F.col("total_presentes")
                / F.col("total_registros")
            ) * 100,
            2
        )
    )

    .withColumn(
        "taxa_alfabetizacao",
        F.round(
            (
                F.col("total_alfabetizados")
                / F.col("total_registros")
            ) * 100,
            2
        )
    )

    .withColumn(
        "taxa_preenchimento",
        F.round(
            (
                F.col(
                    "total_cadernos_preenchidos"
                )
                / F.col("total_registros")
            ) * 100,
            2
        )
    )

    .withColumn(
        "proficiencia_media",
        F.round(
            (
                F.col(
                    "soma_proficiencia_ponderada"
                )
                / F.col("total_registros")
            ),
            2
        )
    )

    .orderBy("ano")
)

In [ ]:
evolucao_alunos_pd = evolucao_alunos.toPandas()

evolucao_alunos_pd

,ano,total_registros,total_presentes,total_alfabetizados,total_cadernos_preenchidos,soma_proficiencia_ponderada,taxa_presenca,taxa_alfabetizacao,taxa_preenchimento,proficiencia_media
0,2023,1747439,1503058,877427,1502809,NaN,86.01,50.21,86.00,NaN
1,2024,2120560,1852788,1107119,1851852,NaN,87.37,52.21,87.33,NaN


In [ ]:
evolucao_taxas_long = (
    evolucao_alunos_pd
    .melt(
        id_vars=["ano"],
        value_vars=[
            "taxa_presenca",
            "taxa_alfabetizacao",
            "taxa_preenchimento"
        ],
        var_name="indicador",
        value_name="percentual"
    )
)

evolucao_taxas_long["indicador"] = (
    evolucao_taxas_long["indicador"]
    .replace({
        "taxa_presenca": "Presença",
        "taxa_alfabetizacao": "Alfabetização",
        "taxa_preenchimento": "Preenchimento do caderno"
    })
)

fig_evolucao_alunos = px.line(
    evolucao_taxas_long,
    x="ano",
    y="percentual",
    color="indicador",
    markers=True,
    title="Evolução dos indicadores agregados dos alunos"
)

fig_evolucao_alunos.update_layout(
    xaxis_title="Ano",
    yaxis_title="Percentual"
)

fig_evolucao_alunos.show()

In [ ]:
fig_proficiencia = go.Figure()

fig_proficiencia.add_trace(
    go.Scatter(
        x=evolucao_alunos_pd["ano"],
        y=evolucao_alunos_pd[
            "proficiencia_media"
        ],
        mode="lines+markers",
        name="Proficiência média"
    )
)

fig_proficiencia.add_hline(
    y=743,
    line_dash="dash",
    annotation_text="Ponto de corte: 743"
)

fig_proficiencia.update_layout(
    title="Proficiência média ponderada dos alunos",
    xaxis_title="Ano",
    yaxis_title="Proficiência"
)

fig_proficiencia.show()

In [ ]:
situacao_proficiencia_pd = (
    gold_perfil_alunos
    .filter(
        F.col("ano") == ANO_REFERENCIA
    )
    .groupBy(
        "situacao_proficiencia"
    )
    .agg(
        F.count("*").alias(
            "grupos_avaliados"
        ),
        F.sum(
            "total_registros"
        ).alias(
            "alunos_representados"
        )
    )
    .toPandas()
)

fig_situacao_proficiencia = px.bar(
    situacao_proficiencia_pd,
    x="situacao_proficiencia",
    y="alunos_representados",
    text="alunos_representados",
    title=(
        "Alunos representados por situação "
        f"de proficiência — {ANO_REFERENCIA}"
    )
)

fig_situacao_proficiencia.update_layout(
    xaxis_title="Situação",
    yaxis_title="Registros de alunos representados"
)

fig_situacao_proficiencia.show()

In [ ]:
alunos_rede = (
    gold_perfil_alunos
    .filter(
        F.col("ano") == ANO_REFERENCIA
    )
    .groupBy("rede")
    .agg(
        F.sum(
            "total_registros"
        ).alias(
            "total_registros"
        ),

        F.sum(
            "total_presentes"
        ).alias(
            "total_presentes"
        ),

        F.sum(
            "total_alfabetizados"
        ).alias(
            "total_alfabetizados"
        )
    )
    .withColumn(
        "taxa_presenca",
        F.round(
            (
                F.col("total_presentes")
                / F.col("total_registros")
            ) * 100,
            2
        )
    )
    .withColumn(
        "taxa_alfabetizacao",
        F.round(
            (
                F.col("total_alfabetizados")
                / F.col("total_registros")
            ) * 100,
            2
        )
    )
)

alunos_rede_pd = alunos_rede.toPandas()

In [ ]:
alunos_rede_long = alunos_rede_pd.melt(
    id_vars=["rede"],
    value_vars=[
        "taxa_presenca",
        "taxa_alfabetizacao"
    ],
    var_name="indicador",
    value_name="percentual"
)

alunos_rede_long["indicador"] = (
    alunos_rede_long["indicador"]
    .replace({
        "taxa_presenca": "Presença",
        "taxa_alfabetizacao": "Alfabetização"
    })
)

fig_alunos_rede = px.bar(
    alunos_rede_long,
    x="rede",
    y="percentual",
    color="indicador",
    barmode="group",
    title=(
        "Presença e alfabetização por rede "
        f"— {ANO_REFERENCIA}"
    )
)

fig_alunos_rede.update_layout(
    xaxis_title="Rede",
    yaxis_title="Percentual"
)

fig_alunos_rede.show()

In [ ]:
graficos = {
    "01_distribuicao_municipal":
        fig_distribuicao_municipal,

    "02_ranking_ufs":
        fig_ranking_uf,

    "03_comparacao_meta_uf":
        fig_comparacao_uf,

    "04_evolucao_alunos":
        fig_evolucao_alunos,

    "05_proficiencia_alunos":
        fig_proficiencia,

    "06_situacao_proficiencia":
        fig_situacao_proficiencia,

    "07_indicadores_alunos_rede":
        fig_alunos_rede
}

for nome, figura in graficos.items():
    caminho = (
        f"{DASHBOARD_PATH}/{nome}.html"
    )

    figura.write_html(
        caminho,
        include_plotlyjs="cdn"
    )

    print(f"Gráfico salvo: {nome}")

Gráfico salvo: 01_distribuicao_municipal
Gráfico salvo: 02_ranking_ufs
Gráfico salvo: 03_comparacao_meta_uf
Gráfico salvo: 04_evolucao_alunos
Gráfico salvo: 05_proficiencia_alunos
Gráfico salvo: 06_situacao_proficiencia
Gráfico salvo: 07_indicadores_alunos_rede


In [ ]:
kpis_pd.to_csv(
    f"{DASHBOARD_PATH}/kpis_dashboard.csv",
    index=False
)

ranking_uf_pd.to_csv(
    f"{DASHBOARD_PATH}/ranking_ufs.csv",
    index=False
)

comparacao_uf_pd.to_csv(
    f"{DASHBOARD_PATH}/comparacao_meta_uf.csv",
    index=False
)

evolucao_alunos_pd.to_csv(
    f"{DASHBOARD_PATH}/evolucao_alunos.csv",
    index=False
)

situacao_proficiencia_pd.to_csv(
    (
        f"{DASHBOARD_PATH}/"
        "situacao_proficiencia.csv"
    ),
    index=False
)

alunos_rede_pd.to_csv(
    f"{DASHBOARD_PATH}/alunos_por_rede.csv",
    index=False
)

print("Tabelas auxiliares exportadas.")

Tabelas auxiliares exportadas.


In [ ]:
print("=" * 70)
print("DASHBOARD ANALYTICS ATUALIZADO")
print("=" * 70)

print(f"Ano de referência: {ANO_REFERENCIA}")
print(
    "Municípios analisados:",
    formatar_numero(
        total_municipios,
        0
    )
)
print(
    "UFs analisadas:",
    formatar_numero(
        total_ufs,
        0
    )
)
print(
    "Escolas representadas:",
    formatar_numero(
        total_escolas,
        0
    )
)
print(
    "Registros de alunos representados:",
    formatar_numero(
        total_alunos_representados,
        0
    )
)
print(
    "Taxa ponderada de presença:",
    f"{formatar_numero(taxa_presenca)}%"
)
print(
    "Taxa ponderada de alfabetização:",
    f"{formatar_numero(taxa_alfabetizacao_alunos)}%"
)
print(
    "Proficiência média ponderada:",
    formatar_numero(proficiencia_media)
)

print()
print(
    f"Arquivos exportados para: "
    f"{DASHBOARD_PATH}"
)

DASHBOARD ANALYTICS ATUALIZADO
Ano de referência: 2024
Municípios analisados: 5.516
UFs analisadas: 25
Escolas representadas: 42.497
Registros de alunos representados: 2.120.560
Taxa ponderada de presença: 87,37%
Taxa ponderada de alfabetização: 52,21%
Proficiência média ponderada: 0,00

Arquivos exportados para: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard


In [ ]:
fig_metas = px.line(
    trajetoria_metas,
    x="ano",
    y="meta",
    markers=True,
    text="meta",
    title="Trajetória planejada da meta nacional até 2030"
)

fig_metas.update_traces(
    textposition="top center"
)

fig_metas.update_layout(
    xaxis_title="Ano",
    yaxis_title="Meta de alfabetização (%)"
)

fig_metas.show()

In [ ]:
quantidade_total = (
    municipio_dashboard
    .select("id_municipio")
    .distinct()
    .count()
)

if quantidade_total >= 100:
    municipio_ranking = municipio_dashboard
    rede_ranking = "Total"
else:
    municipio_ranking = (
        gold_municipio
        .filter(
            (col("ano") == ano_mais_recente)
            & (col("categoria_rede") == "Municipal")
        )
    )

    rede_ranking = "Municipal"

print(
    f"Rede utilizada no ranking municipal: {rede_ranking}"
)

Rede utilizada no ranking municipal: Total


In [ ]:
ranking_municipio_pd = (
    municipio_ranking
    .filter(
        col("indice_geral").isNotNull()
    )
    .select(
        "id_municipio",
        "taxa_alfabetizacao",
        "media_portugues",
        "indice_geral",
        "nivel_desempenho"
    )
    .orderBy(
        desc("indice_geral")
    )
    .limit(10)
    .toPandas()
)

ranking_municipio_pd[
    "id_municipio"
] = ranking_municipio_pd[
    "id_municipio"
].astype(str)

ranking_municipio_pd

,id_municipio,taxa_alfabetizacao,media_portugues,indice_geral,nivel_desempenho
0,2925931,75.47,760.66,418.07,Bom
1,2920304,70.27,760.63,415.45,Bom
2,2923035,73.17,756.30,414.73,Bom
3,2914208,65.38,755.59,410.49,Regular
4,2919405,65.48,754.53,410.01,Regular
5,2929602,64.52,752.92,408.72,Regular
6,2904506,60.98,749.71,405.35,Regular
7,2921807,60.00,749.71,404.86,Regular
8,2902302,60.00,749.68,404.84,Regular
9,2931053,61.80,747.55,404.67,Regular


In [ ]:
fig_ranking_municipio = px.bar(
    ranking_municipio_pd.sort_values(
        "indice_geral",
        ascending=True
    ),
    x="indice_geral",
    y="id_municipio",
    orientation="h",
    text="indice_geral",
    hover_data={
        "taxa_alfabetizacao": ":.2f",
        "media_portugues": ":.2f",
        "nivel_desempenho": True
    },
    title=(
        f"Top 10 municípios por índice geral "
        f"— rede {rede_ranking} — {ano_mais_recente}"
    )
)

fig_ranking_municipio.update_traces(
    textposition="outside"
)

fig_ranking_municipio.update_layout(
    xaxis_title="Índice geral",
    yaxis_title="Código do município"
)

fig_ranking_municipio.show()

In [ ]:
graficos = {
    "01_evolucao_nacional": fig_evolucao,
    "02_ranking_ufs": fig_ranking_uf,
    "03_distribuicao_desempenho": fig_distribuicao,
    "04_taxa_meta_uf": fig_comparacao_uf,
    "05_status_meta_uf": fig_status_uf,
    "06_trajetoria_metas": fig_metas,
    "07_ranking_municipios": fig_ranking_municipio
}

for nome, figura in graficos.items():
    caminho = f"{DASHBOARD_PATH}/{nome}.html"

    figura.write_html(
        caminho,
        include_plotlyjs="cdn"
    )

    print(f"Salvo: {caminho}")

Salvo: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard/01_evolucao_nacional.html
Salvo: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard/02_ranking_ufs.html
Salvo: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard/03_distribuicao_desempenho.html
Salvo: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard/04_taxa_meta_uf.html
Salvo: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard/05_status_meta_uf.html
Salvo: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard/06_trajetoria_metas.html
Salvo: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard/07_ranking_municipios.html


In [ ]:
ranking_uf_pd.to_csv(
    f"{DASHBOARD_PATH}/ranking_ufs.csv",
    index=False
)

ranking_municipio_pd.to_csv(
    f"{DASHBOARD_PATH}/ranking_municipios.csv",
    index=False
)

comparacao_uf_pd.to_csv(
    f"{DASHBOARD_PATH}/comparacao_meta_uf.csv",
    index=False
)

kpis_pd.to_csv(
    f"{DASHBOARD_PATH}/kpis_dashboard.csv",
    index=False
)

print("Tabelas auxiliares exportadas")

Tabelas auxiliares exportadas


In [ ]:
print("=" * 70)
print("DASHBOARD ANALYTICS CONCLUÍDO")
print("=" * 70)

print(f"Ano de referência: {ano_mais_recente}")
print(f"Municípios analisados: {int(total_municipios)}")
print(f"UFs analisadas: {int(total_ufs)}")
print(
    "Média de alfabetização:",
    f"{media_alfabetizacao:.2f}%"
)
print(
    "Média de Português:",
    f"{media_portugues:.2f}"
)
print()
print(f"Arquivos exportados para: {DASHBOARD_PATH}")

DASHBOARD ANALYTICS CONCLUÍDO
Ano de referência: 2024
Municípios analisados: 5516
UFs analisadas: 25
Média de alfabetização: 57.36%
Média de Português: 752.12

Arquivos exportados para: /content/drive/MyDrive/TechChallenge_Fase2/outputs/dashboard
